In [2]:
import numpy as np

# S = 1237 -> d1=1, d2=2, d3=3, d4=7
X = np.array([
    [1, 2],
    [3, 7],
    [5, 5]
], dtype=float) / 5.0  # x1=[0.2, 0.4], x2=[0.6, 1.4], x3=[1.0, 1.0]

# Q = K = V = X
scores = (X @ X.T) / np.sqrt(2)
exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attention_matrix = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
output = attention_matrix @ X

print("3x3 Attention Matrix:")
print(np.round(attention_matrix, 4))
print("\nOutput Matrix:")
print(np.round(output, 4))

3x3 Attention Matrix:
[[0.268  0.3763 0.3556]
 [0.1485 0.4737 0.3778]
 [0.1567 0.4217 0.4217]]

Output Matrix:
[[0.635  0.9897]
 [0.6917 1.1003]
 [0.706  1.0747]]


### Part F1: Scaled Dot-Product Attention (`S = 1237`)

* **Inputs:** $x_1 = [0.2, 0.4]$, $x_2 = [0.6, 1.4]$, $x_3 = [1.0, 1.0]$ ($Q = K = V = X$, $d_k = 2$).
* **3×3 Attention Matrix:**
  $$\begin{bmatrix} 0.2680 & 0.3763 & 0.3556 \\ 0.1485 & 0.4737 & 0.3778 \\ 0.1567 & 0.4217 & 0.4217 \end{bmatrix}$$
* **Output Matrix:**
  $$\begin{bmatrix} 0.6350 & 0.9897 \\ 0.6917 & 1.1003 \\ 0.7060 & 1.0747 \end{bmatrix}$$
* **Key Takeaway:** Token 1 pays the most attention to **Token 2** (`0.3763`) because $x_1$ and $x_2$ have the highest dot product ($0.2(0.6) + 0.4(1.4) = 0.68$), giving a larger scaled score before softmax.

In [4]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")

samples = [
    # 10 English sentences (Karachi places & food)
    ("The biryani at Burns Road was not bad at all, actually quite flavorful.", "POSITIVE", "English"),
    ("Clifton beach at sunset felt peaceful despite the light evening breeze.", "POSITIVE", "English"),
    ("We waited an hour at Do Darya only to get cold, tasteless karahi.", "NEGATIVE", "English"),
    ("Oh great, another two-hour traffic jam near Saddar, just what I needed!", "NEGATIVE", "English"),
    ("The chai at Boat Basin was amazing, though the noisy crowd ruined the mood.", "NEGATIVE", "English"),
    ("Frere Hall gardens are surprisingly clean and relaxing on Sunday mornings.", "POSITIVE", "English"),
    ("I would not say the nihari was terrible, it was actually pretty decent.", "POSITIVE", "English"),
    ("Loved the view from Port Grand, hated paying so much for stale snacks.", "NEGATIVE", "English"),
    ("Tariq Road shopping during Eid week is pure chaos and zero fun.", "NEGATIVE", "English"),
    ("The bun kebab outside our university gate never disappoints.", "POSITIVE", "English"),

    # 10 Roman Urdu / Mixed sentences (Karachi places & food)
    ("Burns Road ki biryani bilkul bhi buri nahi thi, maza aa gaya.", "POSITIVE", "Roman Urdu"),
    ("Wah bhai, Shahrah-e-Faisal ka traffic dekh kar dil khush ho gaya, kya hi baat hai!", "NEGATIVE", "Roman Urdu"),
    ("Do Darya ka khana acha tha lekin service itni slow thi ke mood kharab ho gaya.", "NEGATIVE", "Roman Urdu"),
    ("Clifton beach par thandi hawa aur garam chai ne din bana diya.", "POSITIVE", "Roman Urdu"),
    ("Tariq Road par parking dhoondna kisi saza se kam nahi hai.", "NEGATIVE", "Roman Urdu"),
    ("Boat Basin ke parathay itne bhi bure nahi hain jitna log kehte hain.", "POSITIVE", "Roman Urdu"),
    ("Kya zabardast burger tha, ek bite khate hi pait dard shuru ho gaya!", "NEGATIVE", "Roman Urdu"),
    ("Frere Hall ka mahol kaafi sukoon wala hai, zaroor jana chahiye.", "POSITIVE", "Roman Urdu"),
    ("Saddar ki garmi aur dhool ne poora trip barbaad kar diya.", "NEGATIVE", "Roman Urdu"),
    ("Nihari ka zaiqa kamaal tha magar thandi roti ne sab kharab kar diya.", "NEGATIVE", "Roman Urdu"),
]

results = []
for text, true_label, lang in samples:
    out = clf(text)[0]
    pred_label = out["label"]
    score = round(out["score"], 4)
    results.append((text, lang, true_label, pred_label, score))

# Print table & separate accuracies
eng = [r for r in results if r[1] == "English"]
ur = [r for r in results if r[1] == "Roman Urdu"]

eng_acc = sum(r[2] == r[3] for r in eng) / len(eng)
ur_acc = sum(r[2] == r[3] for r in ur) / len(ur)

print(f"English Accuracy: {eng_acc:.2f} ({sum(r[2] == r[3] for r in eng)}/10)")
print(f"Roman Urdu Accuracy: {ur_acc:.2f} ({sum(r[2] == r[3] for r in ur)}/10)\n")

for i, (text, lang, true_l, pred_l, sc) in enumerate(results, 1):
    mark = "OK" if true_l == pred_l else "WRONG"
    print(f"{i:2d}. [{lang}] True={true_l} | Pred={pred_l} ({sc}) [{mark}] -> {text}")

# Inspect 3 wrong predictions with tokenizer
print("\n--- 3 Wrong Predictions & Tokens ---")
wrong = [r for r in results if r[2] != r[3]][:3]
for text, lang, true_l, pred_l, sc in wrong:
    tokens = clf.tokenizer.tokenize(text)
    print(f"\nSentence: {text}")
    print(f"True: {true_l} | Model: {pred_l} ({sc})")
    print("Tokens:", tokens)

C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_

English Accuracy: 0.70 (7/10)
Roman Urdu Accuracy: 0.50 (5/10)

 1. [English] True=POSITIVE | Pred=POSITIVE (0.9993) [OK] -> The biryani at Burns Road was not bad at all, actually quite flavorful.
 2. [English] True=POSITIVE | Pred=POSITIVE (0.9986) [OK] -> Clifton beach at sunset felt peaceful despite the light evening breeze.
 3. [English] True=NEGATIVE | Pred=NEGATIVE (0.9996) [OK] -> We waited an hour at Do Darya only to get cold, tasteless karahi.
 4. [English] True=NEGATIVE | Pred=POSITIVE (0.9912) [WRONG] -> Oh great, another two-hour traffic jam near Saddar, just what I needed!
 5. [English] True=NEGATIVE | Pred=POSITIVE (0.9904) [WRONG] -> The chai at Boat Basin was amazing, though the noisy crowd ruined the mood.
 6. [English] True=POSITIVE | Pred=POSITIVE (0.9998) [OK] -> Frere Hall gardens are surprisingly clean and relaxing on Sunday mornings.
 7. [English] True=POSITIVE | Pred=POSITIVE (0.9997) [OK] -> I would not say the nihari was terrible, it was actually pretty decent

Part F2 (S = 1237)

Accuracy: English 70% (7/10) | Roman Urdu 50% (5/10) | Overall 60% (12/20)

3 Wrong Cases:

#4 (NEG → POS, 0.9912): ['oh', 'great', ',', 'another', 'two', '-', 'hour', 'traffic', 'jam', 'near', 'sad', '##dar', ',', 'just', 'what', 'i', 'needed', '!'] — Sarcasm missed ("great", "!").

#5 (NEG → POS, 0.9904): ['the', 'cha', '##i', 'at', 'boat', 'basin', 'was', 'amazing', ',', 'though', 'the', 'noisy', 'crowd', 'ruined', 'the', 'mood', '.'] — "amazing" beat "though... ruined".

#8 (NEG → POS, 0.9978): ['loved', 'the', 'view', 'from', 'port', 'grand', ',', 'hated', 'paying', 'so', 'much', 'for', 'stale', 'snacks', '.'] — Opening "loved" outweighed "hated"/"stale".

Why It Fails & Fix: English-only SST-2 training hurts Roman Urdu (50%) and sarcasm/mixed reviews. Fix: switch to a multilingual model (e.g., xlm-roberta-base) and fine-tune on Roman Urdu/mixed reviews.